### Load the Predicted and Reference Abstracted Code

In [26]:
import json
f = open('actual_pred_stripped.json')
comp_result = json.load(f)
f.close()


### Testing for Perfect Prediciton

In [28]:
perfect_pred_count = 0
perfect_pred_testCases = []
for i in comp_result:
    if comp_result[i]['actual'] == comp_result[i]['predicted']:
        perfect_pred_count += 1
        perfect_pred_testCases.append(i)
    else:
        pass

In [ ]:
print(f"Number of Perfect Predictions: {perfect_pred_count}")

Number of Perfect Predictions: 150


In [41]:
testCases_perfectPred = dict()
for i in perfect_pred_testCases:
    testCases_perfectPred[i] = {'actual' : comp_result[i]['actual'], 'predicted' : comp_result[i]['predicted']}

In [42]:
with open('perfect_predictions.json', 'w') as f: 
    json.dump(testCases_perfectPred, f)

### Calculating BELU-4 Score

In [43]:
from datasets import load_metric
import numpy as np

In [48]:
bleu = load_metric('bleu', trust_remote_code=True)
bleu4_testCases = dict()
for i in comp_result:
    predictions = [comp_result[i]['predicted'].split()]
    references = [[comp_result[i]['actual'].split()]]
    bleu4_testCases[i] = bleu.compute(predictions=predictions, references=references)['precisions'][-1]

In [49]:
with open('BLEU-4 Test Scores.json', 'w') as f: 
    json.dump(bleu4_testCases, f)

In [50]:
mean_bleu4 = round(np.mean(list(bleu4_testCases.values())),4)
median_bleu4 = round(np.median(list(bleu4_testCases.values())),4)
std_bleu4 = round(np.std(list(bleu4_testCases.values())),4)

In [51]:
print(f"mean_bleu4: {mean_bleu4}\nmedian_bleu4: {median_bleu4}\nstd_bleu4:{std_bleu4}")

mean_bleu4: 0.7602
median_bleu4: 0.825
std_bleu4:0.2288


### Calculating the Levenshtein Distance

In [52]:
def Levenshtein_Distance(prediction: str, oracle: str) -> int:
    grid = [[float("inf")] * (len(oracle) + 1) for i in range(len(prediction) + 1)]
    
    for i in range(len(oracle) + 1):
        grid[len(prediction)][i] = len(oracle) - i
    for j in range(len(prediction) + 1):
        grid[j][len(oracle)] = len(prediction) - j
        
    for i in range(len(prediction) - 1, -1, -1):
        for j in range(len(oracle) - 1, -1, -1):
            if prediction[i] == oracle[j]:
                grid[i][j] = grid[i+1][j+1]
            else:
                grid[i][j] = 1 + min(grid[i+1][j], grid[i][j+1], grid[i+1][j+1])
    return grid[0][0]

In [53]:
LevDis = dict()
LevDis_norm = dict()
for i in comp_result:
    p = comp_result[i]['predicted']
    a = comp_result[i]['actual']
    LevDis[i] = Levenshtein_Distance(p,a)
    LevDis_norm[i] = Levenshtein_Distance(p,a) / (len(p) if len(p)>len(a) else len(a))

In [54]:
with open('Levenshtein Distance Scores.json', 'w') as f:
    json.dump(LevDis, f)
    
with open('Levenshtein Distance Normalized Scores.json', 'w') as f:
    json.dump(LevDis_norm, f)

In [55]:
mean_levDis = round(np.mean(list(LevDis_norm.values())),4)
median_levDis = round(np.median(list(LevDis_norm.values())),4)
std_levDisf = round(np.std(list(LevDis_norm.values())),4)

In [56]:
print(f"mean_levDis: {mean_levDis}\nmedian_levDis: {median_levDis}\nstd_levDisf:{std_levDisf}")

mean_levDis: 0.2645
median_levDis: 0.2595
std_levDisf:0.17
